# 🚀 SilentExec: Production QLoRA + GGUF + GitHub Release Pipeline

This notebook trains a compact, zero-chatter OS execution agent for **PC and Android** using **Unsloth** on Google Colab (T4/L4 GPU).

### End-to-End Pipeline Overview:
1. **Configuration**: Choice of base model (default: `Qwen2.5-0.5B-Instruct` or `Qwen2.5-1.5B-Instruct`), dataset sizing, quantization method (`q4_k_m`).
2. **Environment Setup**: Unsloth installation and repo synchronization.
3. **Dataset Generation**: 100% schema-verified bilingual training data generation (`schemas/action_schema.json`).
4. **QLoRA SFT Training**: Fast 4-bit LoRA fine-tuning with ChatML formatting.
5. **Real Golden Benchmark Eval**: Greedy decoding on `evals/golden_eval.jsonl` with full Rust SafetyGate oracle scoring.
6. **GGUF Quantization & Checksums**: 4-bit quantization and SHA256 checksum generation.
7. **GitHub Releases Distribution**: Automated asset publishing via `gh release` without committing binaries to Git.


In [ ]:
# ==========================================
# 1. SilentExec Training & Release Configuration
# ==========================================
import os
from pathlib import Path

# Base model choice:
# High-capacity model (strong bilingual Arabic/English multi-step reasoning):
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
# Compact alternative:
# BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

DATASET_COUNT = 5000
RELEASE_TAG = "v0.3.0-model"
QUANT_METHOD = "q4_k_m"
OUTPUT_DIR = "/content/silent_exec_output"
DATA_DIR = "/content/data"
REPO_DIR = "/content/silent-exec"

print(f"Target Base Model:      {BASE_MODEL}")
print(f"Dataset Count:          {DATASET_COUNT}")
print(f"GGUF Quantization:      {QUANT_METHOD}")
print(f"GitHub Release Tag:     {RELEASE_TAG}")


In [ ]:
# ==========================================
# 2. Environment Setup & Dependency Installation
# ==========================================
import os
import subprocess
import sys

print("Installing Unsloth, TRL, Datasets, and jsonschema...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
    "datasets", "trl", "jsonschema", "peft", "transformers", "accelerate"
], check=True)

# Clone or pull SilentExec repository
REPO_URL = "https://github.com/tahadeab-ai/silent-exec.git"
REPO_DIR = "/content/silent-exec"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} into {REPO_DIR}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Pulling latest main in {REPO_DIR}...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=True)

# Ensure repo root is on sys.path and cwd
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"Active Working Directory: {os.getcwd()}")


In [ ]:
# ==========================================
# 3. Deterministic Bilingual Dataset Generation
# ==========================================
import json
import subprocess
import sys
from pathlib import Path

DATA_DIR = "/content/data"
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

print(f"Generating {DATASET_COUNT} bilingual examples into {DATA_DIR}...")
cmd = [
    sys.executable, "nlp-training/generate_dataset.py",
    "--count", str(DATASET_COUNT),
    "--seed", "42",
    "--out-dir", DATA_DIR
]
subprocess.run(cmd, check=True)

# Inspect sample generated lines
train_file = Path(DATA_DIR) / "train.jsonl"
print(f"\n--- Sample Generated Training Examples ({train_file.name}) ---")
with open(train_file, "r", encoding="utf-8") as f:
    for idx in range(3):
        line = f.readline().strip()
        data = json.loads(line)
        user_msg = data["messages"][1]["content"]
        asst_msg = data["messages"][2]["content"]
        print(f"[{idx + 1}] User: {user_msg}")
        print(f"    Assistant: {asst_msg}")


In [ ]:
# ==========================================
# 4. QLoRA Fine-Tuning with Unsloth & SFTTrainer
# ==========================================
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel, is_bfloat16_supported

max_seq_length = 512
dtype = None  # Auto detection: Float16 or Bfloat16
load_in_4bit = True

print(f"Loading {BASE_MODEL} in 4-bit with max_seq_length={max_seq_length}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("Applying LoRA adapters (r=16, alpha=32, target all linear modules)...")
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    max_seq_length=max_seq_length,
)

# Prepare dataset with chat formatting
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

raw_dataset = load_dataset("json", data_files={
    "train": "/content/data/train.jsonl",
    "validation": "/content/data/val.jsonl"
})

formatted_train = raw_dataset["train"].map(formatting_prompts_func, batched=True)
formatted_val = raw_dataset["validation"].map(formatting_prompts_func, batched=True)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_train,
    eval_dataset=formatted_val,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir=OUTPUT_DIR,
        report_to="none"
    ),
)

print("Starting QLoRA SFT training...")
trainer_stats = trainer.train()
print(f"Training successfully finished: {trainer_stats}")


In [ ]:
# ==========================================
# 5. Greedy Decoding & Real Golden Benchmark Evaluation
# ==========================================
import json
import subprocess
import sys
import torch
from pathlib import Path
from unsloth import FastLanguageModel

# Enable native fast inference mode
FastLanguageModel.for_inference(model)

golden_path = Path("/content/silent-exec/evals/golden_eval.jsonl")
preds_path = Path("/content/preds.jsonl")
eval_report_path = Path("/content/eval_report.json")

print(f"Reading {golden_path} for evaluation...")
with open(golden_path, "r", encoding="utf-8") as f:
    eval_records = [json.loads(line) for line in f if line.strip()]

system_prompt = (
    "You are SilentExec, a zero-chatter OS execution agent. "
    "Emit only a valid ActionCommand JSON matching the strict schema. "
    "No conversational responses."
)

predictions = []
print(f"Generating greedy predictions for {len(eval_records)} test cases...")
for idx, item in enumerate(eval_records):
    prompt_text = item["prompt"]
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt_text}
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=160,
            temperature=0.0,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode assistant tokens only
    generated_tokens = outputs[0][inputs.shape[1]:]
    response_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    try:
        parsed_payload = json.loads(response_text)
    except Exception:
        parsed_payload = {"raw": response_text, "action": "parse_error", "platform": "any", "confidence": 0.0}

    predictions.append(parsed_payload)

with open(preds_path, "w", encoding="utf-8") as f:
    for pred in predictions:
        f.write(json.dumps(pred, ensure_ascii=False) + "\n")

print(f"Saved {len(predictions)} predictions to {preds_path}.")

# Execute run_eval.py against predictions
print("Running evals/run_eval.py benchmark against predictions...")
eval_cmd = [
    sys.executable, "evals/run_eval.py",
    "--gold", str(golden_path),
    "--preds", str(preds_path),
    "--out", str(eval_report_path)
]
subprocess.run(eval_cmd, check=True)

with open(eval_report_path, "r", encoding="utf-8") as f:
    report_summary = json.load(f)
print("\n--- Golden Evaluation Summary ---")
print(f"JSON Validity:   {report_summary.get('json_validity_rate', 0):.2f}%")
print(f"Intent Accuracy: {report_summary.get('intent_accuracy', 0):.2f}%")
print(f"Slot Exact:      {report_summary.get('slot_exact_match', 0):.2f}%")


In [ ]:
# ==========================================
# 6. GGUF Quantization & SHA-256 Checksum Generation
# ==========================================
import hashlib
import os
from pathlib import Path

GGUF_DIR = Path("/content/model_gguf")
GGUF_DIR.mkdir(parents=True, exist_ok=True)

print(f"Exporting model to {QUANT_METHOD} GGUF format via Unsloth...")
model.save_pretrained_gguf(
    str(GGUF_DIR),
    tokenizer,
    quantization_method=QUANT_METHOD
)

# Locate the exported .gguf file
gguf_files = list(GGUF_DIR.glob("*.gguf"))
if not gguf_files:
    raise FileNotFoundError(f"No .gguf file found in {GGUF_DIR}")

primary_gguf = gguf_files[0]
file_size_mb = primary_gguf.stat().st_size / (1024 * 1024)
print(f"Exported GGUF File: {primary_gguf.name} ({file_size_mb:.2f} MB)")

# Generate SHA256SUMS.txt
sha256 = hashlib.sha256()
with open(primary_gguf, "rb") as f:
    for chunk in iter(lambda: f.read(65536), b""):
        sha256.update(chunk)

checksum_val = sha256.hexdigest()
checksum_file = GGUF_DIR / "SHA256SUMS.txt"
with open(checksum_file, "w", encoding="utf-8") as f:
    f.write(f"{checksum_val}  {primary_gguf.name}\n")

print(f"SHA256 Checksum: {checksum_val}")
print(f"Checksum file generated at: {checksum_file}")


In [ ]:
# ==========================================
# 7. Automated GitHub Release Publication
# ==========================================
import os
import subprocess
from pathlib import Path

# Retrieve token from Colab Secrets or environment
github_token = None
try:
    from google.colab import userdata
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass

if not github_token:
    github_token = os.environ.get("GITHUB_TOKEN")

if not github_token:
    print("WARNING: GITHUB_TOKEN not found in google.colab.userdata or environment.")
    print("Please set your GitHub Personal Access Token (with 'repo' scope) to publish releases.")
else:
    os.environ["GH_TOKEN"] = github_token
    subprocess.run(["gh", "auth", "setup-git"], check=False)

    repo_target = "tahadeab-ai/silent-exec"
    eval_report = Path("/content/eval_report.json")
    gguf_files = list(Path("/content/model_gguf").glob("*.gguf"))
    checksum_file = Path("/content/model_gguf/SHA256SUMS.txt")

    if not gguf_files or not checksum_file.exists():
        raise FileNotFoundError("GGUF or checksum file missing for release upload")

    gguf_path = gguf_files[0]
    upload_assets = [str(gguf_path), str(checksum_file)]
    if eval_report.exists():
        upload_assets.append(str(eval_report))

    release_title = f"SilentExec NLP Model {RELEASE_TAG}"
    release_notes = (
        f"Automated Release for {RELEASE_TAG}\n"
        f"- Base Architecture: {BASE_MODEL}\n"
        f"- Quantization: {QUANT_METHOD}\n"
        f"- Training Dataset Count: {DATASET_COUNT}\n"
        f"- File: {gguf_path.name}\n"
        f"- Checksum: {checksum_file.read_text().strip()}\n"
    )

    print(f"Publishing release {RELEASE_TAG} to {repo_target}...")
    create_cmd = [
        "gh", "release", "create", RELEASE_TAG,
        "--repo", repo_target,
        "--title", release_title,
        "--notes", release_notes,
        "--clobber"
    ] + upload_assets

    res = subprocess.run(create_cmd, capture_output=True, text=True)
    if res.returncode == 0:
        print(f"Successfully published release {RELEASE_TAG} to GitHub!")
        print(res.stdout)
    else:
        print(f"Release create noted ({res.stderr.strip()}); updating assets...")
        upload_cmd = ["gh", "release", "upload", RELEASE_TAG, "--repo", repo_target, "--clobber"] + upload_assets
        subprocess.run(upload_cmd, check=True)
        print("Assets uploaded successfully via clobber.")
